In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
!ls "/content/drive/Shareddrives/"

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import joblib
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
BASE_PATH = "/content/drive/MyDrive/prototipo_oliveto_pw/models"

CATEGORY_MODEL_PATH = BASE_PATH + "/category_model.pt"
PRIORITY_MODEL_PATH = BASE_PATH + "/priority_model.pt"
VECTORIZER_PATH = BASE_PATH + "/tfidf_vectorizer.joblib"

In [ ]:
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

In [ ]:
vectorizer = joblib.load(VECTORIZER_PATH)
input_dim = len(vectorizer.get_feature_names_out())

print("TF-IDF features:", input_dim)

TF-IDF features: 124


In [ ]:
cat_model = TicketClassifier(input_dim, 3)
prio_model = TicketClassifier(input_dim, 3)

cat_model.load_state_dict(torch.load(CATEGORY_MODEL_PATH))
prio_model.load_state_dict(torch.load(PRIORITY_MODEL_PATH))

cat_model.eval()
prio_model.eval()

print("✅ Modelli caricati correttamente")


✅ Modelli caricati correttamente


In [ ]:
category_labels = {
    0: "Amministrazione",
    1: "Tecnico",
    2: "Commerciale"
}

priority_labels = {
    0: "Bassa",
    1: "Media",
    2: "Alta"
}


In [ ]:
def top_words(vec, model, n=5):
    weights = model.fc.weight.detach().numpy()
    features = vectorizer.get_feature_names_out()

    idx = vec.nonzero()[1]
    scores = weights[:, idx].sum(axis=0)

    top_idx = np.argsort(np.abs(scores))[-n:]
    return [features[idx[i]] for i in top_idx]


In [ ]:
def predict_ticket(b):
    clear_output(wait=True)

    text = subject_input.value + " " + description_input.value

    if len(text.strip()) == 0:
        print("⚠️ Inserisci un testo")
        return

    vec = vectorizer.transform([text])
    vec = torch.tensor(vec.toarray(), dtype=torch.float32)

    with torch.no_grad():
        cat_pred = torch.argmax(cat_model(vec), dim=1).item()
        prio_pred = torch.argmax(prio_model(vec), dim=1).item()

    print("📌 RISULTATO")
    print("Categoria:", category_labels[cat_pred])
    print("Priorità:", priority_labels[prio_pred])
    print("Parole influenti:", ", ".join(top_words(vec, cat_model)))


In [ ]:
subject_input = widgets.Text(
    description="Oggetto:",
    layout=widgets.Layout(width="80%")
)

description_input = widgets.Textarea(
    description="Descrizione:",
    layout=widgets.Layout(width="80%", height="100px")
)

predict_button = widgets.Button(
    description="Classifica Ticket",
    button_style="primary"
)

predict_button.on_click(predict_ticket)

display(subject_input, description_input, predict_button)


📌 RISULTATO
Categoria: Tecnico
Priorità: Media
Parole influenti: non, alle
